# Week 4 – Werkcollege 6: Visual Maandag — spaghetti-grafieken, kleurcontrast en storytelling

Geen nieuwe inleveropdracht vandaag. Je pakt de toernooidata van Week 3 terug (dezelfde `vergelijk_met_week`-data als in Werkcollege 5) en herontwerpt 'm van een onleesbare veelheid aan lijnen naar een grafiek met één duidelijk verhaal.

## Tijdsindicatie

| Moment | Duur | Onderdeel |
|---|---|---|
| Spaghetti identificeren | 10 min | Alle lijnen ongefilterd plotten |
| Eerste reductie: filteren | 10 min | Terug naar 1 tafel, 1 simulatie |
| Eigen bot uitlichten | 20 min | Opacity en lijndikte als pre-attentieve attributen |
| Enkele tegenstanders vergelijken | 10 min | Selectief meerdere lijnen laten opvallen |
| Actietitel + annotatie op het kantelpunt | 15 min | Het moment vinden waarop het verschil ontstaat |
| Kleurcontrast-check | 10 min | Werkt je grafiek nog in grijswaarden? |
| Peer-vergelijking | 10 min | Feedback van een klasgenoot |


## Deel 1 — Spaghetti identificeren (10 min)

Haal dezelfde toernooidata op als in Werkcollege 5: Week 3 vergeleken met Week 1. Plot deze keer alles, zonder te filteren op tafel of simulatie.


In [ ]:
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

API_URL = "https://poker-analytics-api.onrender.com"
STUDENT_ID = "vul_hier_je_student_id_in"
TOKEN = "vul_hier_je_token_in"

response = requests.get(
    f"{API_URL}/toernooi/3",
    params={"student_id": STUDENT_ID, "vergelijk_met_week": 1},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
toernooi = pd.DataFrame(response.json()["hand_log"])
toernooi["lijn_id"] = toernooi["bot_naam"] + "_tafel" + toernooi["tafel"].astype(str) + "_sim" + toernooi["simulatie"].astype(str)

fig = px.line(toernooi, x="hand_nummer", y="stack", color="bot_naam", line_group="lijn_id")
fig.update_layout(title="Chipstack vs Tijd")
fig.show()


Elke bot speelde meerdere tafels en simulaties, dus dezelfde kleur duikt overal opnieuw op, met een compleet andere lijn. Noem minstens 2 redenen waarom dit onleesbaar is, en waarom "meer kleuren toevoegen" het niet zou oplossen.


In [ ]:
# jouw antwoord hier



## Deel 2 — Eerste reductie: filteren (10 min)

De eenvoudigste manier om ruis te verminderen is niet altijd design — vaak is het filteren. Beperk je tot 1 tafel en 1 simulatie, zoals je in Week 3 ook al deed.

Let op: bots worden per simulatie willekeurig over tafels verdeeld, dus "tafel 0, simulatie 0" bevat niet gegarandeerd jouw eigen bot. Zoek eerst een (tafel, simulatie)-combinatie op waar je eigen bot echt aan meespeelde.


In [ ]:
eigen_bot = f"{STUDENT_ID}__w3"

eigen_rijen = toernooi[toernooi["bot_naam"] == eigen_bot]
if eigen_rijen.empty:
    raise ValueError(f"Geen rijen gevonden voor {eigen_bot} — check of STUDENT_ID klopt en je week 3 al hebt ingeleverd.")

gekozen_tafel, gekozen_simulatie = eigen_rijen[["tafel", "simulatie"]].iloc[0]
een_tafel = toernooi[(toernooi["tafel"] == gekozen_tafel) & (toernooi["simulatie"] == gekozen_simulatie)]

fig = px.line(een_tafel, x="hand_nummer", y="stack", color="bot_naam")
fig.show()


🤔 Dit is al leesbaarder, maar er staan nu mogelijk nog steeds 4-8 bots in. Vanaf hoeveel lijnen in een grafiek zou jij zeggen dat kleur alleen niet meer genoeg is?


## Deel 3 — Eigen bot uitlichten (20 min)

Kleur is niet het enige pre-attentieve attribuut. Lijndikte en transparantie (opacity) werken net zo goed, en zijn hier krachtiger: als je alle tegenstanders dezelfde grijze, dunne, transparante lijn geeft, verdwijnen ze naar de achtergrond zonder dat je 10 kleuren nodig hebt.

`eigen_bot` staat al klaar uit Deel 2. Geef die lijn kleur, dikte én opacity; de rest krijgt alleen grijs.


In [ ]:
fig = go.Figure()

for bot_naam in een_tafel["bot_naam"].unique():
    data_bot = een_tafel[een_tafel["bot_naam"] == bot_naam]
    is_eigen_bot = bot_naam == eigen_bot
    fig.add_trace(go.Scatter(
        x=data_bot["hand_nummer"],
        y=data_bot["stack"],
        mode="lines",
        name=bot_naam,
        line=dict(
            color="#E4572E" if is_eigen_bot else "#B0B0B0",
            width=4 if is_eigen_bot else 1,
        ),
        opacity=1.0 if is_eigen_bot else 0.4,
    ))

fig.show()


🎨 Vergelijk dit met Deel 2. Moest je harder zoeken naar jouw bot in de vorige grafiek, of in deze?


## Deel 4 — Enkele tegenstanders vergelijken (10 min)

Soms wil je niet alleen jezelf laten zien, maar ook je 2 grootste concurrenten: de bot met de hoogste eindstand, en die met de laagste. Bereken dat met pandas (niet hardcoden), en geef die twee elk een eigen, onderscheidende kleur — houd de rest grijs.


In [ ]:
eindstanden = een_tafel.sort_values("hand_nummer").groupby("bot_naam")["stack"].last()
hoogste_bot = eindstanden.idxmax()
laagste_bot = eindstanden.idxmin()

kleur_per_bot = {}
for bot_naam in een_tafel["bot_naam"].unique():
    if bot_naam == eigen_bot:
        kleur_per_bot[bot_naam] = "#E4572E"
    elif bot_naam == hoogste_bot:
        kleur_per_bot[bot_naam] = "#2E86AB"
    elif bot_naam == laagste_bot:
        kleur_per_bot[bot_naam] = "#6A0572"
    else:
        kleur_per_bot[bot_naam] = "#B0B0B0"

# bouw hier de grafiek met kleur_per_bot, zelfde opzet als Deel 3



🤔 Je gebruikt nu 4 betekenisvolle kleuren (inclusief grijs). Zou een 5e kleur voor nog een bot de grafiek beter maken, of wordt het weer te veel losse kleuren zonder duidelijk doel?


## Deel 5 — Actietitel + annotatie op het kantelpunt (15 min)

Zoek het hand-nummer waarop jouw bot voor het eerst de hoogste bot voorbij gaat (of, als dat niet gebeurt, het hand-nummer waarop het verschil met de hoogste bot het grootst wordt). Zet daar een annotatie, en schrijf een titel die dat moment benoemt.


In [ ]:
eigen_data = een_tafel[een_tafel["bot_naam"] == eigen_bot].sort_values("hand_nummer")
hoogste_data = een_tafel[een_tafel["bot_naam"] == hoogste_bot].sort_values("hand_nummer")

verschil = eigen_data.set_index("hand_nummer")["stack"] - hoogste_data.set_index("hand_nummer")["stack"]
kantelpunt_hand = verschil.idxmax()
print("grootste voorsprong bij hand:", kantelpunt_hand, "| verschil:", verschil.max())

# voeg hier een fig.add_annotation() toe op (kantelpunt_hand, stack-waarde), en een actietitel



## Deel 6 — Kleurcontrast-check (10 min)

Ongeveer 1 op de 12 mannen heeft een vorm van kleurenblindheid, meestal rood-groen. Een grafiek die alleen op kleur vertrouwt, kan voor die groep onleesbaar zijn.

Check je eigen kleurkeuze: zet je figuur om naar grijswaarden (of doe het met je ogen dicht een beetje toegeknepen) en kijk of je zonder kleur nog steeds kunt zien wat belangrijk is.


In [ ]:
# tip: lijndikte en opacity (zoals in Deel 3) blijven ook in grijswaarden werken --
# dat is precies waarom je die had toegevoegd naast kleur, niet in plaats van een check achteraf.
# maak hier een grijswaarden-versie van je grafiek uit Deel 5, zonder de kleuren te veranderen
# (verander alleen colorway/template, niet je eigen line-width/opacity logica) en kijk of 'm nog werkt.



🤔 Als je grafiek zonder kleur niet meer werkt, wat zou je toevoegen: een patroon, een label, of iets anders?


## Deel 7 — Peer-vergelijking (10 min)

Wissel je eindresultaat uit met een klasgenoot. Focus deze keer specifiek op: had je zonder de legenda te lezen kunnen zien wie zijn/haar eigen bot is?


In [ ]:
# feedback van je klasgenoot:



---

## Reflectievragen

🎨 Wat werkte beter om je eigen bot te laten opvallen: kleur, lijndikte, of opacity? Of de combinatie?

🤔 In Deel 2 loste filteren al een deel van het probleem op, nog vóór je iets aan kleur deed. Waarom wordt filteren vaak vergeten als "design"-stap?

🎨 Je actietitel in Deel 5 benoemt een specifiek moment. Wat verlies je aan informatie door je op dat ene moment te focussen, en is dat verlies de duidelijkheid waard?

💡 Kleurcontrast-checks worden vaak achteraf gedaan, als de grafiek al bijna klaar is. Wat zou er gebeuren als je die check aan het begin deed, vóór je kleuren kiest?

🤔 Deze grafiek combineerde 4 technieken (filteren, opacity/lijndikte, selectieve kleur, annotatie). Welke had je kunnen weglaten zonder dat de grafiek onleesbaar werd?

---

### Vooruitblik: Week 5

Geopandas, Folium en Streamlit. Bot v3 krijgt een bluf-parameter en een geolocatie.
